In [5]:
import pandas as pd
import re
df=pd.read_parquet("../data/processed/clean_cases.parquet")
print(df.shape)

(26688, 3)


In [2]:
print(df["legal_text"].str.len().describe())

count    2.668800e+04
mean     3.273970e+04
std      5.665167e+04
min      1.620000e+02
25%      1.283175e+04
50%      2.151000e+04
75%      3.609150e+04
max      2.546582e+06
Name: legal_text, dtype: float64


In [3]:
print(
    "HEADNOTE:",
    df["legal_text"]
    .str.contains(
        "HEADNOTE:",
        case=False,
        na=False
    ).sum()
)

print(
    "ACT:",
    df["legal_text"]
    .str.contains(
        "ACT:",
        case=False,
        na=False
    ).sum()
)

print(
    "JUDGMENT:",
    df["legal_text"]
    .str.contains(
        "JUDGMENT",
        case=False,
        na=False
    ).sum()
)

HEADNOTE: 11491
ACT: 12036
JUDGMENT: 25926


In [6]:
def extract_summary(text):

    text = str(text)

    # CASE 1
    headnote = re.search(
        r'HEADNOTE:(.*?)(JUDGMENT|$)',
        text,
        flags=re.DOTALL | re.IGNORECASE
    )

    if headnote:
        return headnote.group(1)[:2000]

    # CASE 2
    act = re.search(
        r'ACT:(.*?)(HEADNOTE|JUDGMENT|$)',
        text,
        flags=re.DOTALL | re.IGNORECASE
    )

    if act:
        return act.group(1)[:2000]

    # CASE 3
    judgment = re.search(
        r'JUDGMENT(.*)',
        text,
        flags=re.DOTALL | re.IGNORECASE
    )

    if judgment:
        return judgment.group(1)[:2000]

    # CASE 4
    return text[:2000]

In [7]:
df["legal_summary"] = (
    df["legal_text"]
    .apply(extract_summary)
)

In [8]:
print(
    df["legal_summary"]
    .str.len()
    .describe()
)

count    26688.000000
mean      1798.777915
std        539.074895
min          1.000000
25%       2000.000000
50%       2000.000000
75%       2000.000000
max       2000.000000
Name: legal_summary, dtype: float64


In [9]:
print(
    "Empty Summaries:",
    (
        df["legal_summary"]
        .str.len() == 0
    ).sum()
)

Empty Summaries: 0


In [10]:
df.to_parquet(
    "../data/processed/summaries.parquet",
    index=False
)

print("Saved")

Saved


In [11]:
print(df.columns)

print(df.shape)

Index(['case_name', 'year', 'legal_text', 'legal_summary'], dtype='object')
(26688, 4)


In [12]:
summary_df = df[
    ["case_name", "year", "legal_summary"]
].copy()

summary_df.to_parquet(
    "../data/processed/summaries_only.parquet",
    index=False
)

print(summary_df.shape)

(26688, 3)


In [13]:
from pathlib import Path

file_path = Path(
    "../data/processed/summaries_only.parquet"
)

print(
    round(
        file_path.stat().st_size /
        (1024*1024),
        2
    ),
    "MB"
)

24.88 MB
